# Notebook 06b — Data Quality & Missingness

**Supplementary notebook** exploring motion-capture data limitations.

Motion-capture systems produce incomplete frames when markers are occluded by the bird's body, pass below the capture volume, or are mislabelled by reconstruction software. The main PCA analysis (NB02) uses only complete 8-marker frames (~126k bilateral). A larger dataset (~559k bilateral frames) includes frames with partial marker visibility.

This notebook explores:
1. **Missingness patterns** — which markers drop out, how often, and together with which others
2. **Coordinate density differences** — where partial-frame markers sit relative to complete frames
3. **Dropout geometry** — where the remaining markers are when a given marker is missing
4. **Mislabelling detection** — evidence that some marker labels are swapped

Cross-reference NB06 (Robustness Validation) for the quantitative tests showing these limitations affect sampling density, not morphospace structure.

In [ ]:
# --- Setup ---
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import yaml

plt.rcParams['font.family'] = 'Andale Mono'
np.set_printoptions(suppress=True, precision=4)

from kinematic_morphospace import filter_by, to_unilateral
from kinematic_morphospace.null_testing import (
    load_missing_marker_dataset,
)
from kinematic_morphospace.plotting import save_figure

MARKER_NAMES = ['wingtip', 'primary', 'secondary', 'tailtip']
FIGURE_DIR = Path('../../figures/supplementary')

## §1 Dataset exploration

In [ ]:
# --- Load the 559k dataset with missing markers ---
missing_path = Path('../../data/raw/labelled_markers_with_missing.npz')
missing_bilateral, missing_info, missing_cols = load_missing_marker_dataset(missing_path)

# Scale to each hawk's maximum wingspan (matching the main analysis pipeline)
with open('../../src/kinematic_morphospace/TotalWingspans.yml') as f:
    total_wingspans = yaml.safe_load(f)

hawk_names = {1: 'Drogon', 2: 'Rhaegal', 3: 'Ruby', 4: 'Toothless', 5: 'Charmander'}
bird_ids = missing_info['BirdID'].astype(int).values
years = missing_info['Year'].astype(int).values

for hawk_id, hawk_name in hawk_names.items():
    for year in [2017, 2020]:
        if year not in total_wingspans.get(hawk_name, {}):
            continue
        mask = (bird_ids == hawk_id) & (years == year)
        if mask.sum() > 0:
            missing_bilateral[mask] /= total_wingspans[hawk_name][year]

# Filter to straight flight and convert to unilateral
straight_missing = (missing_info['Obstacle'].astype(int) == 0).values
missing_straight = missing_bilateral[straight_missing]

left = missing_straight[:, [0, 2, 4, 6], :].copy()
left[:, :, 0] *= -1  # mirror x
right = missing_straight[:, [1, 3, 5, 7], :]
missing_unilateral = np.concatenate([left, right], axis=0)

print(f'559k dataset: {missing_bilateral.shape[0]:,} bilateral frames')
print(f'Straight-flight bilateral: {missing_straight.shape[0]:,}')
print(f'Straight-flight unilateral (both sides): {missing_unilateral.shape[0]:,}')

In [ ]:
# --- Separate complete vs genuinely partial frames ---
any_nan = np.any(np.isnan(missing_unilateral), axis=(1, 2))
complete_data = missing_unilateral[~any_nan]
partial_data = missing_unilateral[any_nan]

print(f'Complete frames (all 4 markers): {complete_data.shape[0]:,}')
print(f'Partial frames (>=1 marker missing): {partial_data.shape[0]:,}')
print(f'Partial fraction: {partial_data.shape[0] / missing_unilateral.shape[0]:.1%}')

In [ ]:
# --- Per-marker dropout rates ---
n_partial = partial_data.shape[0]

print('Marker dropout rates in partial frames:')
print(f'{"Marker":<12} {"Missing":>10} {"Present":>10} {"Dropout %":>10}')
print('-' * 45)
for m, name in enumerate(MARKER_NAMES):
    missing_count = np.isnan(partial_data[:, m, 0]).sum()
    present_count = n_partial - missing_count
    print(f'{name:<12} {missing_count:>10,} {present_count:>10,} {missing_count/n_partial:>10.1%}')

In [ ]:
# --- Co-occurrence of missingness ---
# When marker A is missing, how often is marker B also missing?
missing_mask = np.isnan(partial_data[:, :, 0])  # (N_partial, 4)

# Conditional probability: P(B missing | A missing)
co_occur = np.zeros((4, 4))
for a in range(4):
    a_missing = missing_mask[:, a]
    n_a = a_missing.sum()
    if n_a == 0:
        continue
    for b in range(4):
        co_occur[a, b] = (a_missing & missing_mask[:, b]).sum() / n_a

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(co_occur, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(MARKER_NAMES, rotation=45, ha='right')
ax.set_yticklabels(MARKER_NAMES)
ax.set_xlabel('Marker B')
ax.set_ylabel('Marker A (missing)')
ax.set_title('P(B missing | A missing)')

# Annotate cells
for a in range(4):
    for b in range(4):
        colour = 'white' if co_occur[a, b] > 0.5 else 'black'
        ax.text(b, a, f'{co_occur[a, b]:.2f}', ha='center', va='center',
                fontsize=10, color=colour)

plt.colorbar(im, ax=ax, label='Conditional probability')
fig.tight_layout()
plt.show()

In [ ]:
# --- Per-hawk dropout rates ---
# The unilateral data doubles frames (left + right), so use bilateral info
missing_straight_info = missing_info[straight_missing].copy()

# Per-marker missingness in bilateral data (check left side: indices 0,2,4,6)
bilateral_marker_names = ['wingtip', 'primary', 'secondary', 'tailtip']
bilateral_left_idx = [0, 2, 4, 6]

rows = []
for hawk_id, hawk_name in hawk_names.items():
    for year in [2017, 2020]:
        hmask = (missing_straight_info['BirdID'].astype(int).values == hawk_id) & \
                (missing_straight_info['Year'].astype(int).values == year)
        n_total = hmask.sum()
        if n_total == 0:
            continue
        subset = missing_straight[hmask]
        any_miss = np.any(np.isnan(subset), axis=(1, 2)).sum()
        row = {'Hawk': hawk_name, 'Year': year, 'Frames': n_total,
               'Partial %': any_miss / n_total}
        for m, (name, idx) in enumerate(zip(bilateral_marker_names, bilateral_left_idx)):
            # Check left side marker
            miss_count = np.isnan(subset[:, idx, 0]).sum()
            row[f'{name} %'] = miss_count / n_total
        rows.append(row)

dropout_df = pd.DataFrame(rows)
print(dropout_df.to_string(index=False, float_format='{:.1%}'.format))

## §2 Coordinate densities: complete vs partial frames

For each marker, we compare the spatial distribution of coordinates in complete frames (all 4 unilateral markers present) against partial frames (at least one marker missing, but the marker in question is present). Two projections are shown: x–z (lateral vs vertical, the wing's cross-section) and x–y (lateral vs longitudinal, the wing planform viewed from above).

In [ ]:
def plot_density_comparison(complete_data, partial_data, marker_names,
                            coord_a, coord_b, label_a, label_b,
                            suptitle, bins=60):
    """Plot 2D density comparison: complete vs partial vs difference.

    Parameters
    ----------
    coord_a, coord_b : int
        Coordinate indices (0=x, 1=y, 2=z) for horizontal and vertical axes.
    """
    fig, axes = plt.subplots(len(marker_names), 3, figsize=(15, 4.5 * len(marker_names)))

    for m, name in enumerate(marker_names):
        # Complete frames
        ac = complete_data[:, m, coord_a]
        bc = complete_data[:, m, coord_b]

        # Partial frames where this marker IS present
        present = ~np.isnan(partial_data[:, m, 0])
        ap = partial_data[present, m, coord_a]
        bp = partial_data[present, m, coord_b]

        # Shared range
        a_range = (min(ac.min(), ap.min()), max(ac.max(), ap.max()))
        b_range = (min(bc.min(), bp.min()), max(bc.max(), bp.max()))
        hist_kw = dict(bins=bins, range=[a_range, b_range], density=True)

        H_c, aedges, bedges = np.histogram2d(ac, bc, **hist_kw)
        H_p, _, _ = np.histogram2d(ap, bp, **hist_kw)

        vmax = np.nanquantile(np.concatenate([H_c.ravel(), H_p.ravel()]), 0.95)

        # Complete
        H_plot = H_c.T.copy()
        H_plot[H_plot == 0] = np.nan
        im = axes[m, 0].pcolormesh(aedges, bedges, H_plot, cmap='Blues', vmax=vmax)
        axes[m, 0].set_ylabel(f'{name}\n{label_b}')
        if m == 0:
            axes[m, 0].set_title('Complete (all markers)')
        plt.colorbar(im, ax=axes[m, 0])

        # Partial
        H_plot = H_p.T.copy()
        H_plot[H_plot == 0] = np.nan
        im = axes[m, 1].pcolormesh(aedges, bedges, H_plot, cmap='Purples', vmax=vmax)
        if m == 0:
            axes[m, 1].set_title('Partial (marker present, others missing)')
        plt.colorbar(im, ax=axes[m, 1])

        # Difference
        H_diff = H_p.T - H_c.T
        mask = (H_c.T > 0) | (H_p.T > 0)
        H_diff[~mask] = np.nan
        vabs = np.nanquantile(np.abs(H_diff), 0.98)
        im = axes[m, 2].pcolormesh(aedges, bedges, H_diff, cmap='RdBu_r',
                                   vmin=-vabs, vmax=vabs)
        if m == 0:
            axes[m, 2].set_title('Partial \u2212 Complete')
        plt.colorbar(im, ax=axes[m, 2], label='\u0394 Density')

        for ax in axes[m, :]:
            ax.set_xlabel(label_a)
            ax.set_aspect('equal')

    fig.suptitle(suptitle, y=1.01)
    fig.tight_layout()
    return fig

In [ ]:
# --- x vs z (lateral vs vertical) ---
fig_xz = plot_density_comparison(
    complete_data, partial_data, MARKER_NAMES,
    coord_a=0, coord_b=2,
    label_a='x (lateral)', label_b='z (vertical)',
    suptitle='Marker position densities: complete vs partial frames (x vs z)',
)
save_figure(fig_xz, str(FIGURE_DIR / 'S06b_density_xz.pdf'))
plt.show()

In [ ]:
# --- x vs y (lateral vs longitudinal) ---
fig_xy = plot_density_comparison(
    complete_data, partial_data, MARKER_NAMES,
    coord_a=0, coord_b=1,
    label_a='x (lateral)', label_b='y (longitudinal)',
    suptitle='Marker position densities: complete vs partial frames (x vs y)',
)
save_figure(fig_xy, str(FIGURE_DIR / 'S06b_density_xy.pdf'))
plt.show()

**Interpretation.** Partial frames over-represent folded-wing and flapping-extreme configurations relative to complete frames. Complete frames are concentrated at the gliding configuration — wings level and fully spread, when all markers are most visible to the overhead cameras. During deep wingbeats and wing-folding, distal feather markers tuck behind the body or pass below the capture volume, producing partial frames precisely when the wing is in its most extreme configurations.

The complete-marker dataset therefore overrepresents gliding and spread-wing configurations relative to their true frequency in flight.

## §3 Where are the other markers when one drops out?

For each target marker that is missing, we plot the positions of the *remaining* markers that are still visible. This reveals whether dropout is posture-dependent — e.g. the wingtip disappears when wings are folded.

In [ ]:
def plot_dropout_positions(complete_data, partial_data, marker_names,
                           coord_a, coord_b, label_a, label_b,
                           suptitle, bins=60):
    """For each target marker that is missing, plot positions of the other markers."""
    fig, axes = plt.subplots(len(marker_names), 3,
                             figsize=(15, 4.5 * len(marker_names)))

    for m_target, target_name in enumerate(marker_names):
        others = [i for i in range(len(marker_names)) if i != m_target]

        # Complete frames: target present, extract other markers
        ac = complete_data[:, others, coord_a].ravel()
        bc = complete_data[:, others, coord_b].ravel()

        # Partial frames where target is MISSING but others are present
        target_missing = np.isnan(partial_data[:, m_target, 0])
        others_present = np.all(~np.isnan(partial_data[:, others, 0]), axis=1)
        dropout_mask = target_missing & others_present

        ap = partial_data[dropout_mask][:, others, coord_a].ravel()
        bp = partial_data[dropout_mask][:, others, coord_b].ravel()

        n_dropout = dropout_mask.sum()
        print(f'{target_name} missing (others present): {n_dropout:,} frames')

        if n_dropout < 100:
            for ax in axes[m_target, :]:
                ax.text(0.5, 0.5, 'Too few frames', transform=ax.transAxes, ha='center')
            continue

        # Shared range
        a_range = (min(ac.min(), ap.min()), max(ac.max(), ap.max()))
        b_range = (min(bc.min(), bp.min()), max(bc.max(), bp.max()))
        hist_kw = dict(bins=bins, range=[a_range, b_range], density=True)

        H_c, aedges, bedges = np.histogram2d(ac, bc, **hist_kw)
        H_p, _, _ = np.histogram2d(ap, bp, **hist_kw)

        vmax = np.nanquantile(np.concatenate([H_c.ravel(), H_p.ravel()]), 0.95)

        # Complete
        H_plot = H_c.T.copy()
        H_plot[H_plot == 0] = np.nan
        im = axes[m_target, 0].pcolormesh(aedges, bedges, H_plot, cmap='Blues', vmax=vmax)
        axes[m_target, 0].set_ylabel(f'{target_name} missing\n{label_b}')
        if m_target == 0:
            axes[m_target, 0].set_title('Other markers (complete)')
        plt.colorbar(im, ax=axes[m_target, 0])

        # Dropout frames
        H_plot = H_p.T.copy()
        H_plot[H_plot == 0] = np.nan
        im = axes[m_target, 1].pcolormesh(aedges, bedges, H_plot, cmap='Purples', vmax=vmax)
        if m_target == 0:
            axes[m_target, 1].set_title('Other markers (dropout)')
        plt.colorbar(im, ax=axes[m_target, 1])

        # Difference
        H_diff = H_p.T - H_c.T
        mask = (H_c.T > 0) | (H_p.T > 0)
        H_diff[~mask] = np.nan
        vabs = np.nanquantile(np.abs(H_diff), 0.98)
        im = axes[m_target, 2].pcolormesh(aedges, bedges, H_diff, cmap='RdBu_r',
                                          vmin=-vabs, vmax=vabs)
        if m_target == 0:
            axes[m_target, 2].set_title('Dropout \u2212 Complete')
        plt.colorbar(im, ax=axes[m_target, 2], label='\u0394 Density')

        for ax in axes[m_target, :]:
            ax.set_xlabel(label_a)
            ax.set_aspect('equal')

    fig.suptitle(suptitle, y=1.01)
    fig.tight_layout()
    return fig

In [ ]:
# --- Dropout positions: x vs z ---
fig_drop_xz = plot_dropout_positions(
    complete_data, partial_data, MARKER_NAMES,
    coord_a=0, coord_b=2,
    label_a='x (lateral)', label_b='z (vertical)',
    suptitle='Where does each marker drop out?\nPositions of OTHER markers when target is missing (x vs z)',
)
save_figure(fig_drop_xz, str(FIGURE_DIR / 'S06b_dropout_positions_xz.pdf'))
plt.show()

In [ ]:
# --- Dropout positions: x vs y ---
fig_drop_xy = plot_dropout_positions(
    complete_data, partial_data, MARKER_NAMES,
    coord_a=0, coord_b=1,
    label_a='x (lateral)', label_b='y (longitudinal)',
    suptitle='Where does each marker drop out?\nPositions of OTHER markers when target is missing (x vs y)',
)
save_figure(fig_drop_xy, str(FIGURE_DIR / 'S06b_dropout_positions_xy.pdf'))
plt.show()

**Interpretation.** Marker dropout is posture-dependent, not random. When the wingtip is missing, the remaining markers show the wing in a more folded configuration — distal feather markers tuck behind the body during deep wingbeats. When the tailtip is missing, the remaining wing markers show more extreme tail deflections. This confirms that certain wing configurations preferentially lose markers, and the complete-marker subset undersamples these extreme postures.

## §4 Mislabelling detection

Motion-capture reconstruction software can confuse nearby markers during rapid movement, swapping their labels. Along the wing, the anatomical ordering should always be: **secondary** (most proximal) < **primary** < **wingtip** (most distal) in the lateral (x) direction. Violations of this ordering indicate label swaps.

We test this on frames where the relevant markers are both present.

In [ ]:
# --- Anatomical ordering violations ---
# Expected x-ordering (lateral): secondary < primary < wingtip
# Marker indices: 0=wingtip, 1=primary, 2=secondary, 3=tailtip

# Test on all unilateral frames (complete + partial) where the relevant pair is present
all_data = missing_unilateral

pairs = [
    ('secondary', 'primary', 2, 1),   # secondary.x should be < primary.x
    ('primary', 'wingtip', 1, 0),      # primary.x should be < wingtip.x
    ('secondary', 'wingtip', 2, 0),    # secondary.x should be < wingtip.x
]

print(f'{"Pair":<25} {"Testable":>10} {"Violations":>12} {"Rate":>8}')
print('-' * 58)

violation_masks = {}
for label, (name_inner, name_outer, idx_inner, idx_outer) in enumerate(pairs):
    # Both markers must be present (not NaN)
    inner_present = ~np.isnan(all_data[:, idx_inner, 0])
    outer_present = ~np.isnan(all_data[:, idx_outer, 0])
    testable = inner_present & outer_present

    # Violation: inner marker x >= outer marker x
    x_inner = all_data[testable, idx_inner, 0]
    x_outer = all_data[testable, idx_outer, 0]
    violated = x_inner >= x_outer

    pair_name = f'{name_inner} >= {name_outer}'
    n_testable = testable.sum()
    n_violated = violated.sum()
    print(f'{pair_name:<25} {n_testable:>10,} {n_violated:>12,} {n_violated/n_testable:>8.2%}')

    # Store full-length mask for later plotting
    full_mask = np.zeros(all_data.shape[0], dtype=bool)
    full_mask[testable] = violated
    violation_masks[f'{name_inner}_vs_{name_outer}'] = full_mask

In [ ]:
# --- Break down violations by complete vs partial frames ---
print(f'{"Pair":<25} {"Complete":>12} {"Partial":>12}')
print('-' * 52)
for key, mask in violation_masks.items():
    n_complete_viol = mask[~any_nan].sum()
    n_partial_viol = mask[any_nan].sum()
    n_complete_total = (~any_nan).sum()
    n_partial_total = any_nan.sum()
    print(f'{key:<25} {n_complete_viol:>6,} ({n_complete_viol/n_complete_total:.2%}) '
          f'{n_partial_viol:>6,} ({n_partial_viol/n_partial_total:.2%})')

In [ ]:
# --- Spatial evidence: where do ordering violations occur? ---
# Overlay violation frames on the density plots for the most common swap pair
# (primary vs wingtip, indices 1 and 0)

viol_mask = violation_masks['primary_vs_wingtip']
viol_frames = all_data[viol_mask]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, (coord_a, coord_b, lab_a, lab_b) in enumerate([
    (0, 2, 'x (lateral)', 'z (vertical)'),
    (0, 1, 'x (lateral)', 'y (longitudinal)'),
]):
    for row, (m, name) in enumerate([(0, 'wingtip'), (1, 'primary')]):
        ax = axes[row, col]

        # Background: all frames density
        present = ~np.isnan(all_data[:, m, 0])
        a_all = all_data[present, m, coord_a]
        b_all = all_data[present, m, coord_b]

        ax.hist2d(a_all, b_all, bins=60, cmap='Greys', alpha=0.5,
                  density=True)

        # Overlay: violation frames
        viol_present = viol_mask & present
        a_viol = all_data[viol_present, m, coord_a]
        b_viol = all_data[viol_present, m, coord_b]

        ax.scatter(a_viol, b_viol, c='red', s=1, alpha=0.3, rasterized=True)
        ax.set_xlabel(lab_a)
        ax.set_ylabel(lab_b)
        ax.set_title(f'{name} positions (primary \u2265 wingtip in x)')
        ax.set_aspect('equal')

fig.suptitle('Where do primary\u2013wingtip ordering violations occur?', y=1.02)
fig.tight_layout()
save_figure(fig, str(FIGURE_DIR / 'S06b_ordering_violations.pdf'))
plt.show()

**Interpretation.** Ordering violations — frames where the primary marker appears more lateral than the wingtip — cluster in specific regions of coordinate space. These correspond to the red excess regions visible in the §2 density difference plots, confirming that the density mismatch between complete and partial frames is partly driven by mislabelled markers.

The mislabelling rate quantifies how often the reconstruction software confuses adjacent feather markers during rapid wing movement. Notably, violation rates are slightly higher in complete frames than in partial frames — this likely reflects the labelling pipeline: when fewer markers are visible, the software has fewer candidates to confuse, so label swaps are less common. In the complete 8-marker dataset used for PCA, these ordering violations are present at a low rate and concentrated in extreme wing configurations where adjacent markers are physically closest.

## §5 Summary & limitations

**Key findings:**

- **Marker dropout is posture-dependent.** Distal markers (wingtip, primary) disappear during folded-wing and flapping-extreme configurations, when they tuck behind the body or leave the capture volume. The complete-marker dataset under-represents these extreme postures.
- **Markers co-drop.** When one distal marker is missing, others are more likely to be missing too, reflecting the same postural cause.
- **Mislabelling is detectable.** Anatomical ordering violations reveal that reconstruction software occasionally swaps adjacent feather marker labels, particularly wingtip and primary during rapid movement.
- **Bias is in density, not coverage.** The morphospace is identical regardless of marker visibility — partial frames do not visit regions of wing shape that complete frames miss entirely (see NB06 §8.4d coverage comparison).

**Limitations not addressed here:**

- **Ghost markers** — spurious reflections that create phantom marker detections — are a known motion-capture artefact but are not formally tested in this analysis.
- **Temporal structure of dropout** — whether markers tend to disappear for long stretches or flicker on and off is not explored here (see NB06 §8.6 for temporal autocorrelation of the complete dataset).